# GSE290024 human retinal organoid validation

Controlled 25 mM versus 17-17.5 mM glucose retinal-organoid validation of locked signatures. Final analysis uses edgeR/limma; the failed PyDESeq2 attempt was removed.

**Important:** These notebooks document the finalized analysis logic. Public raw data are downloaded from the cited repositories; large intermediate objects and outputs are intentionally not embedded in the GitHub version. Where a dataset lacks condition-level biological replication, results are explicitly treated as descriptive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, pandas as pd, numpy as np

ROOT = Path("/content/drive/MyDrive/DR_Muller_AdaptiveReactive")
OUT  = ROOT / "05_GSE290024_organoid"
OUT.mkdir(parents=True, exist_ok=True)

with open(ROOT/"signatures/GSE328966_LOCKED_discovery_signatures.json") as f:
    sig = json.load(f)

print("Early184:", len(sig["early_up_transient_all"]))
print("IFN:", len(sig["early_IFN_alpha_beta"]))
print("Retinoid:", len(sig["early_retinoid_metabolism"]))
print("ECM/GAG:", len(sig["early_ECM_GAG_exploratory"]))

In [ ]:
!pip -q install GEOparse

import GEOparse, pandas as pd

gse = GEOparse.get_GEO("GSE290024", destdir=".", silent=True)

meta = pd.DataFrame([{
    "GSM": g.name,
    "title": g.metadata["title"][0],
    "treatment": next(
        (x.split(": ",1)[1] for x in g.metadata.get("characteristics_ch1",[])
         if x.startswith("treatment:")), "control")
} for g in gse.gsms.values()])

display(meta)

print("Samples:", len(meta))
print("Supplementary:", gse.metadata.get("supplementary_file", []))

In [ ]:
rows = []

for g in gse.gsms.values():
    x = {"GSM": g.name, "title": g.metadata["title"][0]}
    for s in g.metadata.get("characteristics_ch1", []):
        if ": " in s:
            k, v = s.split(": ", 1)
            x[k] = v
    rows.append(x)

meta = pd.DataFrame(rows)

display(meta.T)

In [ ]:
for g in gse.gsms.values():
    print("\n", g.name, "|", g.metadata["title"][0])
    for k in ["source_name_ch1","growth_protocol_ch1",
              "treatment_protocol_ch1","description"]:
        if k in g.metadata:
            print(k, ":", " ".join(g.metadata[k]))

In [ ]:
url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE290nnn/GSE290024/suppl/GSE290024_RAW_counts.txt.gz"

counts = pd.read_csv(url, sep="\t", compression="gzip")

print("Shape:", counts.shape)
print("Columns:", counts.columns.tolist())
display(counts.head())

In [ ]:
X = counts.set_index("Gene")

print("Genes:", X.shape[0])
print("Duplicate IDs:", X.index.duplicated().sum())
print("All-zero genes:", (X.sum(axis=1) == 0).sum())
print("Negative counts:", (X < 0).sum().sum())

qc = pd.DataFrame({
    "library_M": X.sum(axis=0) / 1e6,
    "detected_genes": (X > 0).sum(axis=0)
})

display(qc.round(2))

In [ ]:
from sklearn.decomposition import PCA

cpm = X.div(X.sum(axis=0), axis=1) * 1e6
keep = (cpm >= 1).sum(axis=1) >= 3
logcpm = np.log2(cpm.loc[keep] + 0.5)

print("Genes retained:", keep.sum())

print("\nSample correlations:")
display(logcpm.corr().round(3))

pca = PCA(n_components=2)
pc = pd.DataFrame(
    pca.fit_transform(logcpm.T),
    index=logcpm.columns,
    columns=["PC1","PC2"]
)

print("Variance explained:", np.round(pca.explained_variance_ratio_*100,1))
display(pc.round(2))

In [ ]:
corr = logcpm.corr()

for grp, cols in {
    "Control": ["Control_R1","Control_R2","Control_R3"],
    "D-glucose": ["D-glucose_R1","D-glucose_R2","D-glucose_R3"]
}.items():
    vals = [corr.loc[a,b] for i,a in enumerate(cols) for b in cols[i+1:]]
    print(grp, "within-group correlations:",
          np.round(vals,3),
          "| mean =", round(np.mean(vals),3))

print("\nD-glucose_R3 correlations:")
print(corr["D-glucose_R3"].round(3).to_string())

In [ ]:
!R -q -e 'if(!requireNamespace("BiocManager",quietly=TRUE)) install.packages("BiocManager",repos="https://cloud.r-project.org"); if(!requireNamespace("edgeR",quietly=TRUE)) BiocManager::install("edgeR",ask=FALSE,update=FALSE)'

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv["cnt"] = X.astype(int)

ro.r(r'''
library(edgeR)

group <- factor(c(rep("Control",3), rep("Dglucose",3)))
y <- DGEList(counts=cnt, group=group)

keep <- filterByExpr(y, group=group)
y <- y[keep,,keep.lib.sizes=FALSE]
y <- normLibSizes(y)

design <- model.matrix(~group)
fit <- glmQLFit(y, design)
q <- glmQLFTest(fit, coef=2)

de_org <- topTags(q, n=Inf)$table
de_org$Gene <- rownames(de_org)

cat("Genes tested:", nrow(de_org), "\n")
cat("FDR < 0.05:", sum(de_org$FDR < .05), "\n")
cat("UP:", sum(de_org$FDR < .05 & de_org$logFC > 0), "\n")
cat("DOWN:", sum(de_org$FDR < .05 & de_org$logFC < 0), "\n")
''')

In [ ]:
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

with localconverter(ro.default_converter + pandas2ri.converter):
    de_org = ro.conversion.rpy2py(ro.r("de_org"))

de_org = de_org.reset_index(drop=True)

print(de_org.shape)
display(de_org.head())

In [ ]:
!pip -q install mygene

import mygene
mg = mygene.MyGeneInfo()

ann = pd.DataFrame(
    mg.querymany(
        de_org["Gene"].tolist(),
        scopes="ensembl.gene",
        fields="symbol",
        species="human",
        verbose=False
    )
)

ann = (ann[["query","symbol"]]
       .dropna()
       .drop_duplicates("query")
       .rename(columns={"query":"Gene"}))

de_org2 = de_org.merge(ann, on="Gene", how="left")

print("Mapped:", de_org2["symbol"].notna().sum(), "/", len(de_org2))
print("Unique symbols:", de_org2["symbol"].nunique())

In [ ]:
orth = pd.read_csv(
    ROOT/"02_Human_scRNA_2026/Mouse_to_Human_orthologs_LOCKED.csv"
)

rows = []

for name in ["early_up_transient_all",
             "early_IFN_alpha_beta",
             "early_retinoid_metabolism",
             "early_ECM_GAG_exploratory"]:

    hg = (orth[orth["mouse_symbol"].isin(sig[name])]
          ["human_symbol"].dropna().unique())

    present = [g for g in hg if g in set(de_org2["symbol"])]

    rows.append({
        "signature": name,
        "mouse_n": len(sig[name]),
        "human_orthologs": len(hg),
        "tested_here": len(present)
    })

coverage = pd.DataFrame(rows)
display(coverage)

In [ ]:
from rpy2.robjects.vectors import StrVector

# choose one expressed Ensembl ID per human symbol
best = (de_org2.dropna(subset=["symbol"])
        .sort_values("logCPM", ascending=False)
        .drop_duplicates("symbol"))

sig_ids = {}
for name in ["early_up_transient_all","early_IFN_alpha_beta",
             "early_retinoid_metabolism","early_ECM_GAG_exploratory"]:
    hg = orth.loc[orth.mouse_symbol.isin(sig[name]), "human_symbol"].unique()
    sig_ids[name] = best.loc[best.symbol.isin(hg), "Gene"].tolist()
    ro.globalenv[name] = StrVector(sig_ids[name])

ro.r(r'''
library(limma)

design <- model.matrix(~group)
v <- voom(y, design, plot=FALSE)

sets <- list(
 Early184=early_up_transient_all,
 IFN6=early_IFN_alpha_beta,
 Retinoid5=early_retinoid_metabolism,
 ECM13=early_ECM_GAG_exploratory
)

ans <- do.call(rbind, lapply(names(sets), function(nm) {
    z <- camera(v, sets[[nm]], design, contrast=2)
    data.frame(signature=nm, NGenes=z$NGenes,
               Direction=z$Direction, PValue=z$PValue)
}))

ans$FDR <- p.adjust(ans$PValue, "BH")
ans
''')

In [ ]:
with localconverter(ro.default_converter + pandas2ri.converter):
    ans_df = ro.conversion.rpy2py(ro.r("ans"))

ans_df = ans_df.reset_index(drop=True)
print(ans_df.to_string(index=False))

In [ ]:
ro.r(r'''
Z <- t(scale(t(v$E)))

S <- sapply(sets, function(g)
    colMeans(Z[intersect(g, rownames(Z)), , drop=FALSE])
)

S <- t(S)

fitS <- eBayes(lmFit(S, design))
tt <- topTable(fitS, coef=2, number=Inf, sort.by="none")

direct <- data.frame(
    signature=rownames(tt),
    effect=tt$logFC,
    p=tt$P.Value
)
direct$FDR <- p.adjust(direct$p, "BH")

direct
''')

In [ ]:
with localconverter(ro.default_converter + pandas2ri.converter):
    direct_df = ro.conversion.rpy2py(ro.r("direct"))

direct_df = direct_df.reset_index(drop=True)
print(direct_df.to_string(index=False))

In [ ]:
rows = []

for name, ids in sig_ids.items():
    x = de_org2[de_org2["Gene"].isin(ids)]

    rows.append({
        "signature": name,
        "n": len(x),
        "median_logFC": x["logFC"].median(),
        "mean_logFC": x["logFC"].mean(),
        "up_genes": (x["logFC"] > 0).sum(),
        "down_genes": (x["logFC"] < 0).sum(),
        "FDR05": (x["FDR"] < .05).sum()
    })

sig_gene_summary = pd.DataFrame(rows)
display(sig_gene_summary.round(3))

In [ ]:
focus = ["early_IFN_alpha_beta",
         "early_retinoid_metabolism",
         "early_ECM_GAG_exploratory"]

for name in focus:
    x = de_org2[de_org2["Gene"].isin(sig_ids[name])][
        ["symbol","logFC","PValue","FDR"]
    ].sort_values("FDR")

    print("\n", name)
    print(x.round(4).to_string(index=False))

x = de_org2[de_org2["Gene"].isin(sig_ids["early_up_transient_all"])]

print("\nEarly184 FDR<0.05:",
      (x.FDR < .05).sum(),
      "| up:", ((x.FDR < .05) & (x.logFC > 0)).sum(),
      "| down:", ((x.FDR < .05) & (x.logFC < 0)).sum())

In [ ]:
from scipy.stats import binomtest, fisher_exact

rows = []

for name, ids in sig_ids.items():
    x = de_org2[de_org2["Gene"].isin(ids)]
    n = len(x)
    up = (x.logFC > 0).sum()
    nde = (x.FDR < .05).sum()

    bg_de = (de_org2.FDR < .05).sum()
    bg_n  = len(de_org2)

    p_up = binomtest(up, n, .5, alternative="greater").pvalue

    p_de = fisher_exact([
        [nde, n-nde],
        [bg_de-nde, (bg_n-n)-(bg_de-nde)]
    ], alternative="greater").pvalue

    rows.append([name,n,up,nde,p_up,p_de])

final_check = pd.DataFrame(rows, columns=[
    "signature","n","up","FDR05","p_up_bias","p_DE_enrichment"
])

display(final_check.round(4))

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)

ans_df.to_csv(OUT/"GSE290024_CAMERA_locked_signatures.csv", index=False)
direct_df.to_csv(OUT/"GSE290024_direct_signature_tests.csv", index=False)
sig_gene_summary.to_csv(OUT/"GSE290024_signature_gene_summary.csv", index=False)
final_check.to_csv(OUT/"GSE290024_final_enrichment_checks.csv", index=False)

de_org2.to_csv(OUT/"GSE290024_edgeR_DE_all_genes.csv", index=False)

print("GSE290024 ORGANOID ARM CLOSED")